# PG Data Service — API Demo

This notebook demonstrates how Christopher (or any approved consumer) accesses Performance Golf data through the data service API.

**What the API provides:**
- `list_cards()` — see what enriched card views are available
- `list_datasets()` — see what raw datasets are available
- `get_card(card_name, date_from, date_to)` — enriched data with Beast Modes computed, PII stripped
- `get_raw(dataset, date_from, date_to)` — raw data, PII stripped

**Governance:** PII is always stripped. No escape hatch. Only datasets in `datasets.yaml` are queryable.

In [ ]:
# Setup — point Python at the data service directory
import os, sys

os.chdir(os.path.dirname(os.path.abspath("__file__")))
# If running from outside the pg-data-service directory, uncomment and set:
# sys.path.insert(0, "/path/to/pg-data-service")

from api import get_card, get_raw, list_cards, list_datasets

## 1. What datasets can I access?

In [ ]:
list_datasets()

## 2. Raw data access

Pull raw rows from any approved dataset. PII columns are automatically stripped — you'll never see emailAddress, names, addresses, payment info, etc.

In [ ]:
# Fetch raw data — last 90 days, 500 rows for demo
raw_df = get_raw("ad_performance", "2025-12-15", "2026-03-15", limit=500)
print(f"{len(raw_df)} rows, {len(raw_df.columns)} columns")
raw_df.head()

In [ ]:
# Confirm PII columns are NOT in the output
pii_check = ["emailAddress", "customerId", "firstName", "lastName", "ipAddress",
             "phoneNumber", "shipFirstName", "shipLastName", "cardLast4"]
found_pii = [c for c in pii_check if c in raw_df.columns]
if found_pii:
    print(f"WARNING: PII columns found: {found_pii}")
else:
    print("PII check passed — no PII columns in output")

In [ ]:
# See all available columns
print(f"Columns ({len(raw_df.columns)}):")
for i, col in enumerate(sorted(raw_df.columns), 1):
    print(f"  {i:3d}. {col}")

## 3. Enriched data (Beast Modes)

This runs the full pipeline: fetch → aggregate by ad → compute Beast Mode metrics (Net ROAS, CPA, NC Net ROAS, etc.). One row per ad.

In [ ]:
# Enriched pipeline — full Beast Mode calculations (one row per ad)
summary_df = get_card("ad_performance_summary", "2025-12-15", "2026-03-15")
print(f"{len(summary_df)} ads")
summary_df.head()

In [ ]:
# Portfolio summary
print(f"Total ads: {len(summary_df)}")
print(f"Total spend: ${summary_df['spend'].sum():,.2f}")
print(f"Total revenue: ${summary_df['gross_revenue'].sum():,.2f}")
print(f"Portfolio Gross ROAS: {summary_df['gross_revenue'].sum() / summary_df['spend'].sum():.2f}x")

In [ ]:
# Top 10 ads by spend
display_cols = ["ad_name", "funnel", "spend", "gross_revenue", "net_roas"]
cols = [c for c in display_cols if c in summary_df.columns]
summary_df.sort_values("spend", ascending=False)[cols].head(10)

## 4. Error handling

The API enforces the allowlist. Requesting a dataset that isn't approved gets a clear error.

In [ ]:
# Try requesting a dataset that's not approved
try:
    get_raw("some_random_dataset", "2026-01-01", "2026-03-15")
except KeyError as e:
    print(f"Blocked: {e}")

## 5. Quick analysis — your playground

Use the DataFrames above however you want. Filter, group, export to CSV — the data is clean and PII-free.

In [ ]:
# Example: spend by funnel
if "funnel" in summary_df.columns:
    funnel_summary = summary_df.groupby("funnel").agg(
        ads=("ad_name", "count"),
        total_spend=("spend", "sum"),
        total_revenue=("gross_revenue", "sum"),
    ).sort_values("total_spend", ascending=False)
    funnel_summary["roas"] = funnel_summary["total_revenue"] / funnel_summary["total_spend"]
    funnel_summary

In [ ]:
# Example: export to CSV for your own pipeline
# summary_df.to_csv("my_export.csv", index=False)
# raw_df.to_csv("my_raw_export.csv", index=False)